In [2]:
import pandas as pd

# 读取 parquet 文件
df = pd.read_parquet("ar-00000-of-00001.parquet", engine="pyarrow")  # 或 engine="fastparquet"

# 保存为 CSV
df.to_csv("ar-00000-of-00001.csv", index=False)

print("转换完成，已保存为 ar-00000-of-00001.csv")

转换完成，已保存为 ar-00000-of-00001.csv


In [1]:
import pandas as pd
import csv

# ========== 去重配置 ==========
DEDUP_KEEP = 'first'   # 重复推文保留第一条

# ========== 文件配置列表 ==========
file_configs = [
    {
        "path": "all_data_amh.tsv",            # 改成你的实际文件名
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal", # Normal -> 0，Hate/Abusive -> 1
        "header": 0
    },
    {
        "path": "all_data_hau.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_orm.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_tir.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_yor.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "test_swa.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_arq.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_ibo.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_pcm.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_twi.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_zul.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "train_swa.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_ary.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_kin.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_som.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    {
        "path": "all_data_xho.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "label",
        "label_normal_value": "Normal",
        "header": 0
    },
    # 在这里按相同格式添加更多文件 ...
]

# ========== 工具函数（与你的代码完全相同）==========
def read_file_with_fallback(file_config):
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_toxicity_label(df, config):
    # 根据你的需求，使用 label_normal_value 模式
    if "label_column" in config:
        col = config["label_column"]
        if col not in df.columns:
            print(f"  ✗ 未找到 label_column: {col}")
            return pd.Series([-1] * len(df))
        raw = df[col]
        if "label_normal_value" in config:
            normal_vals = config["label_normal_value"]
            if isinstance(normal_vals, str):
                normal_vals = [normal_vals]
            label_series = pd.Series(1, index=df.index)   # 默认有毒
            mask_normal = raw.isin(normal_vals)
            label_series[mask_normal] = 0                 # Normal 设为 0
            print(f"  ✓ 使用 label_normal_value={normal_vals}：这些值 -> 0，其他值 -> 1")
            return label_series
        # 其他模式这里不会用到
    print(f"  ⚠ 无 label_column → 标签全为-1")
    return pd.Series([-1] * len(df))

def extract_tweets_and_labels(df, config):
    tweet_col = config["tweet_column"]
    if tweet_col not in df.columns:
        print(f"  ✗ 未找到推文列 '{tweet_col}'，实际列名: {df.columns.tolist()}")
        return [], []
    non_empty = df[tweet_col].notna()
    valid_df = df[non_empty].copy()
    tweets = valid_df[tweet_col].tolist()
    full_labels = extract_toxicity_label(df, config)
    labels = full_labels[non_empty].tolist()
    labels = [str(l) for l in labels]
    valid_01_count = sum(1 for l in labels if l in ('0','1'))
    print(f"  提取到 {len(tweets)} 条推文，其中有效标签（0/1）数: {valid_01_count}")
    return tweets, labels

def main():
    all_tweets = []
    all_labels = []

    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets, labels = extract_tweets_and_labels(df, config)
        all_tweets.extend(tweets)
        all_labels.extend(labels)

    raw_df = pd.DataFrame({'tweet': all_tweets, 'label': all_labels})
    raw_output = "all_datasets_with_labels.csv"
    raw_df.to_csv(raw_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 原始合并: {len(raw_df)} 条记录，已保存至 {raw_output}")

    dedup_df = raw_df.drop_duplicates(subset=['tweet'], keep=DEDUP_KEEP)
    removed = len(raw_df) - len(dedup_df)
    print(f"✓ 去重后: {len(dedup_df)} 条记录（移除 {removed} 条重复）")

    dedup_output = "all_datasets_with_labels_dedup.csv"
    dedup_df.to_csv(dedup_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"✓ 去重结果已保存至 {dedup_output}")

    print("\n去重后前5条预览：")
    print(dedup_df.head())

if __name__ == "__main__":
    main()


处理文件 1: all_data_amh.tsv
  ✓ 使用编码 utf-8 成功读取 all_data_amh.tsv
  ✓ 使用 label_normal_value=['Normal']：这些值 -> 0，其他值 -> 1
  提取到 4958 条推文，其中有效标签（0/1）数: 4958

处理文件 2: all_data_hau.tsv
  ✓ 使用编码 utf-8 成功读取 all_data_hau.tsv
  ✓ 使用 label_normal_value=['Normal']：这些值 -> 0，其他值 -> 1
  提取到 6644 条推文，其中有效标签（0/1）数: 6644

处理文件 3: all_data_orm.tsv
  ✓ 使用编码 utf-8 成功读取 all_data_orm.tsv
  ✓ 使用 label_normal_value=['Normal']：这些值 -> 0，其他值 -> 1
  提取到 5032 条推文，其中有效标签（0/1）数: 5032

处理文件 4: all_data_tir.tsv
  ✓ 使用编码 utf-8 成功读取 all_data_tir.tsv
  ✓ 使用 label_normal_value=['Normal']：这些值 -> 0，其他值 -> 1
  提取到 5072 条推文，其中有效标签（0/1）数: 5072

处理文件 5: all_data_yor.tsv
  ✓ 使用编码 utf-8 成功读取 all_data_yor.tsv
  ✓ 使用 label_normal_value=['Normal']：这些值 -> 0，其他值 -> 1
  提取到 4906 条推文，其中有效标签（0/1）数: 4906

处理文件 6: test_swa.tsv
  ✓ 使用编码 utf-8 成功读取 test_swa.tsv
  ✓ 使用 label_normal_value=['Normal']：这些值 -> 0，其他值 -> 1
  提取到 3168 条推文，其中有效标签（0/1）数: 3168

处理文件 7: all_data_arq.tsv
  ✓ 使用编码 utf-8 成功读取 all_data_arq.tsv
  ✓ 使用 label_normal_value=['Norma